# 05a — Sampling: NUTS without Hessian initialization (DATA)

**Hypothesis H1**: Let NUTS adapt the mass matrix from identity, with no
Hessian information whatsoever. Long warmup gives adaptation enough draws
for 910-dimensional covariance estimation.

**This notebook produces sampling artifacts.** Analysis lives in
`05a_sampling_no_hessian_analysis.ipynb`.

**Outputs** (per `RUN_NAME`):
- `data/results_no_hessian/{RUN_NAME}/nuts_samples_no_hessian.pt` — chains, adapted M, step size
- `data/results_no_hessian/{RUN_NAME}/iterative/round_NN.pt` — H1-1 per-round artifacts
- `data/results_no_hessian/{RUN_NAME}/iterative/round_summary_raw.json` — H1-1 loop summary


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

# Resolve project root
cwd = Path.cwd().resolve()
project_root = next(
    (
        p
        for p in (cwd, *cwd.parents)
        if (p / "packages" / "pytorch_models" / "markov_transformer.py").exists()
    ),
    None,
)
if project_root is None:
    alt_root = cwd / "projects" / "markov-chain-learning"
    if (alt_root / "packages" / "pytorch_models" / "markov_transformer.py").exists():
        project_root = alt_root

if project_root is None:
    raise RuntimeError("Could not locate markov-chain-learning project root")

packages_dir = project_root / "packages"
if str(packages_dir) not in sys.path:
    sys.path.insert(0, str(packages_dir))

from pytorch_models import MarkovTransformer

DATA_DIR = project_root / "experiments" / "single-chain" / "data"
print(f"Project root: {project_root}")
print(f"Data dir: {DATA_DIR}")

Project root: /Users/ashrafahmed/code/slt-deep/projects/markov-chain-learning
Data dir: /Users/ashrafahmed/code/slt-deep/projects/markov-chain-learning/experiments/single-chain/data


In [3]:
# ── Papermill parameters ──
RUN_NAME: str = "default"

# H1 (initial warmup + production)
SIGMA_PRIOR: float = 10.0
START_FRESH: bool = True
N_WARMUP: int = 2000  # single warmup before production
N_PRODUCTION: int = 1000  # production samples (cold chain)
MAX_TREE_DEPTH: int = 4
TARGET_ACCEPT: float = 0.69

# H1-1 (iterative adaptation refinement)
RUN_ITERATIVE: bool = True
N_ROUNDS: int = 5
N_WARMUP_PER_ROUND: int = 2000
N_SAMPLES_PER_ROUND: int = 1000
RESUME: bool = False  # resume from highest existing round_NN.pt


In [4]:
# Parameters
RUN_NAME = "default"
N_WARMUP = 2000
N_PRODUCTION = 500
RUN_ITERATIVE = "true"
N_ROUNDS = 5
N_WARMUP_PER_ROUND = 2000
N_SAMPLES_PER_ROUND = 500
MAX_TREE_DEPTH = 4
TARGET_ACCEPT = 0.6
SIGMA_PRIOR = 10.0
START_FRESH = "true"
RESUME = "false"


## Load data & model from checkpoint

In [5]:
# Load dataset
data = torch.load(DATA_DIR / "sequences.pt", weights_only=False)
sequences = data["sequences"]
data_cfg = data["config"]

VOCAB_SIZE = int(data_cfg["n_states"])
MAX_LEN = int(data_cfg["L"])
PAD_ID = int(data_cfg.get("pad_id", -1))
DGP_REGIME = data_cfg.get("dgp_regime", "unknown")

print(f"DGP regime: {DGP_REGIME}")
print(
    f"Sequences: {tuple(sequences.shape)}, VOCAB_SIZE={VOCAB_SIZE}, MAX_LEN={MAX_LEN}"
)

DGP regime: single
Sequences: (3000, 10), VOCAB_SIZE=5, MAX_LEN=10


In [6]:
# Load trained model from checkpoint
device = torch.device("cpu")
D_MODEL = VOCAB_SIZE * 2

model = MarkovTransformer(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    max_len=MAX_LEN,
).to(device)

ckpt = torch.load(DATA_DIR / "checkpoint_single_chain.pt", weights_only=False)
model.load_state_dict(ckpt["model_state"])
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"Loaded checkpoint (epoch {ckpt['epoch']}, val_loss={ckpt['val_loss']:.4f})")
print(f"Total parameters: {total_params:,}")

Loaded checkpoint (epoch 73, val_loss=1.2806)
Total parameters: 910


In [7]:
# Prepare data tensors for sampling
x_data = sequences[:, :-1].to(device)
y_data = sequences[:, 1:].to(device)
x_data = x_data.clone()
x_data[x_data == PAD_ID] = 0

mle_param = torch.cat([p.flatten() for p in model.parameters()]).detach()
print(f"MLE parameter vector: d = {mle_param.shape[0]}")

MLE parameter vector: d = 910


## Sampling setup

In [8]:
def loss_fn(logits, targets):
    ce = F.cross_entropy(
        logits.reshape(-1, VOCAB_SIZE),
        targets.reshape(-1),
        reduction="none",
        ignore_index=PAD_ID,
    )
    ce = ce.view(logits.shape[:-1])
    mask = (targets != PAD_ID).float()
    return (ce * mask).sum() / mask.sum()


def make_prior_logp(mu: torch.Tensor, sigma=10.0):
    """Gaussian prior N(mu, sigma^2 I) — centred at the MAP."""
    mean = mu.detach().clone()

    def prior_logp(params):
        flat = torch.cat([p.flatten() for p in params])
        diff = flat - mean
        return -0.5 * diff.pow(2).sum() / (sigma**2)

    return prior_logp


## NUTS — initial warmup + production

Single parametrized warmup followed by production sampling with fixed M.


In [9]:
import collections

from torch_bdn.bn import BayesianNet
from torch_bdn.sampling import NUTS, Perturb, Sampler

# Per-run output directory
RESULTS_DIR = DATA_DIR / "results_no_hessian" / RUN_NAME
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MASS_MATRIX_PATH = RESULTS_DIR / "adapted_mass_matrix.pt"
ITER_DIR = RESULTS_DIR / "iterative"


def print_mass_matrix_diagnostics(M: torch.Tensor, name: str = "M"):
    """Print conditioning and structure diagnostics for a mass matrix."""
    M_cpu = M.detach().cpu().float()
    eigvals_M = torch.linalg.eigvalsh(M_cpu)
    sv = torch.linalg.svdvals(M_cpu)
    cond = float(sv[0] / sv[-1]) if sv[-1] > 0 else float("inf")
    print(
        f"{name}: shape={tuple(M_cpu.shape)}  "
        f"eig=[{float(eigvals_M.min()):.4e}, {float(eigvals_M.max()):.4e}]  "
        f"cond={cond:.2e}"
    )


if START_FRESH:
    mass_matrix = None
    print(f"Starting from IDENTITY mass matrix (d={mle_param.shape[0]})")
else:
    if MASS_MATRIX_PATH.exists():
        mass_matrix = torch.load(MASS_MATRIX_PATH, weights_only=True)
        print(f"Loaded mass matrix from {MASS_MATRIX_PATH}")
        print_mass_matrix_diagnostics(mass_matrix, name="Loaded M")
    else:
        mass_matrix = None
        print(f"No saved mass matrix at {MASS_MATRIX_PATH}, starting from identity")

bn = BayesianNet(
    model, loss_fn, make_prior_logp(mle_param, sigma=SIGMA_PRIOR), compile=True
)
print(f"Run name: {RUN_NAME}")
print(f"Results dir: {RESULTS_DIR}")


Starting from IDENTITY mass matrix (d=910)


Run name: default
Results dir: /Users/ashrafahmed/code/slt-deep/projects/markov-chain-learning/experiments/single-chain/data/results_no_hessian/default


In [10]:
# ── Warmup (adapts mass matrix from identity or loaded M) ──
warmup_result = Sampler(bn, x_data, y_data).sample(
    config=NUTS(
        n_warmup=N_WARMUP,
        step_size=0.01,
        max_tree_depth=MAX_TREE_DEPTH,
        target_accept=TARGET_ACCEPT,
        mass_matrix=mass_matrix,
        adapt_mass_matrix=True,
    ),
    n_samples=1,
    n_chains=1,
    init_strategy=Perturb(scale=0.1),
    n_cores=1,
)

diag = warmup_result.chains[0].diagnostics
adapted_M = diag.get("adapted_mass_matrix")
adapted_step_size = diag.get("adapted_step_size", diag["step_size"])

if adapted_M is not None:
    mass_matrix = adapted_M
    torch.save(adapted_M, MASS_MATRIX_PATH)
    print(f"✓ Warmup complete. Saved mass matrix to {MASS_MATRIX_PATH}")
    print_mass_matrix_diagnostics(adapted_M, name="Adapted M")
    print(f"  adapted ε = {adapted_step_size:.4e}")
    print(f"  accept = {warmup_result.chains[0].acceptance_rate:.3f}")
else:
    print("⚠ No adapted mass matrix returned")


  [NUTS warmup] step 10/2001  ε=1.56e-02  depth=4 (hit max)  L=15  α=0.58  divs=2/10  mass=identity


  [NUTS warmup] step 20/2001  ε=4.37e-02  depth=4 (hit max)  L=15  α=0.54  divs=0/10  mass=identity


  [NUTS warmup] step 30/2001  ε=7.85e-02  depth=4 (hit max)  L=15  α=0.62  divs=1/10  mass=identity


  [NUTS warmup] step 40/2001  ε=1.63e-02  depth=4 (hit max)  L=15  α=0.61  divs=0/10  mass=identity


  [NUTS warmup] step 50/2001  ε=5.63e-03  depth=4 (hit max)  L=15  α=0.56  divs=0/10  mass=identity


...

  [NUTS warmup] step 1970/2001  ε=9.70e-03  depth=4 (hit max)  L=15  α=0.61  divs=0/10  mass=full


  [NUTS warmup] step 1980/2001  ε=4.87e-04  depth=4 (hit max)  L=15  α=0.57  divs=0/10  mass=full


  [NUTS warmup] step 1990/2001  ε=2.09e-03  depth=4 (hit max)  L=15  α=0.64  divs=0/10  mass=full


  [NUTS warmup] step 2000/2001  ε=7.23e-03  depth=4 (hit max)  L=15  α=0.54  divs=0/10  mass=full


✓ Warmup complete. Saved mass matrix to /Users/ashrafahmed/code/slt-deep/projects/markov-chain-learning/experiments/single-chain/data/results_no_hessian/default/adapted_mass_matrix.pt
Adapted M: shape=(910, 910)  eig=[1.5975e-05, 6.4076e+00]  cond=4.04e+05
  adapted ε = 2.1184e-03
  accept = 1.000


In [11]:
# ── Production sampling: fixed mass matrix, no further adaptation ──
mass_matrix_prod = torch.load(MASS_MATRIX_PATH, weights_only=True)
print(f"Loaded mass matrix from {MASS_MATRIX_PATH}")

result = Sampler(bn, x_data, y_data).sample(
    config=NUTS(
        n_warmup=0,
        step_size=adapted_step_size,
        max_tree_depth=MAX_TREE_DEPTH,
        target_accept=TARGET_ACCEPT,
        mass_matrix=mass_matrix_prod,
        adapt_mass_matrix=False,
    ),
    n_samples=N_PRODUCTION,
    n_chains=1,
    init_strategy=Perturb(scale=0.1),
    n_cores=1,
)

for ci, ch in enumerate(result.chains):
    diag_p = ch.diagnostics
    tree_depths = diag_p.get("tree_depths", [])
    print(
        f"Chain {ci}: accept={ch.acceptance_rate:.3f}  "
        f"ε={diag_p.get('step_size', 0):.4e}  "
        f"mean_depth={diag_p.get('mean_tree_depth', 0):.1f}  "
        f"divergences={diag_p.get('n_divergences', 0)}/{len(tree_depths)}"
    )


Loaded mass matrix from /Users/ashrafahmed/code/slt-deep/projects/markov-chain-learning/experiments/single-chain/data/results_no_hessian/default/adapted_mass_matrix.pt


  [NUTS sample] step 10/500  ε=2.12e-03  depth=4 (hit max)  L=15  α=1.00  divs=0/10  mass=full


  [NUTS sample] step 20/500  ε=2.12e-03  depth=4 (hit max)  L=15  α=1.00  divs=0/10  mass=full


  [NUTS sample] step 30/500  ε=2.12e-03  depth=4 (hit max)  L=15  α=1.00  divs=0/10  mass=full


  [NUTS sample] step 40/500  ε=2.12e-03  depth=4 (hit max)  L=15  α=1.00  divs=0/10  mass=full


...

  [NUTS sample] step 460/500  ε=2.12e-03  depth=4 (hit max)  L=15  α=1.00  divs=0/10  mass=full


  [NUTS sample] step 470/500  ε=2.12e-03  depth=4 (hit max)  L=15  α=1.00  divs=0/10  mass=full


  [NUTS sample] step 480/500  ε=2.12e-03  depth=4 (hit max)  L=15  α=1.00  divs=0/10  mass=full


  [NUTS sample] step 490/500  ε=2.12e-03  depth=4 (hit max)  L=15  α=1.00  divs=0/10  mass=full


  [NUTS sample] step 500/500  ε=2.12e-03  depth=4 (hit max)  L=15  α=1.00  divs=0/10  mass=full
Chain 0: accept=1.000  ε=2.1184e-03  mean_depth=4.0  divergences=0/500


In [12]:
# ── Persist production samples ──
nuts_path = RESULTS_DIR / "nuts_samples_no_hessian.pt"

chains_payload = [
    {
        "parameters": torch.stack(ch.parameters).cpu(),
        "acceptance_rate": ch.acceptance_rate,
        "diagnostics": ch.diagnostics,
    }
    for ch in result.chains
]

torch.save(
    {
        "chains": chains_payload,
        "config": {
            "run_name": RUN_NAME,
            "n_chains": 1,
            "n_warmup": N_WARMUP,
            "n_samples": N_PRODUCTION,
            "sigma_prior": SIGMA_PRIOR,
            "dgp_regime": DGP_REGIME,
            "mass_matrix_init": "identity" if START_FRESH else "loaded",
            "adapt_mass_matrix": True,
            "max_tree_depth": MAX_TREE_DEPTH,
            "target_accept": TARGET_ACCEPT,
        },
        "mle_param": mle_param.cpu(),
        "adapted_M": adapted_M.cpu() if adapted_M is not None else None,
        "adapted_step_size": float(adapted_step_size),
        "mass_matrix_path": str(MASS_MATRIX_PATH),
        "hessian_path": str(DATA_DIR / "hessian.pt"),
    },
    nuts_path,
)
print(f"✓ Saved to {nuts_path}")
print(f"  1 chain × {N_PRODUCTION} samples × d={mle_param.shape[0]}")


✓ Saved to /Users/ashrafahmed/code/slt-deep/projects/markov-chain-learning/experiments/single-chain/data/results_no_hessian/default/nuts_samples_no_hessian.pt
  1 chain × 500 samples × d=910


## H1-1: Iterative Adaptation Refinement

**Hypothesis H1-1**: Starting from H1's adapted mass matrix, run repeated
rounds of (warmup -> production -> use new M). Each round saves a
checkpoint so the analysis notebook can recompute ACF/ESS without rerunning.


In [13]:
# ── H1-1 setup ──
import json as _json

if RUN_ITERATIVE:
    ITER_DIR.mkdir(parents=True, exist_ok=True)

    # Determine starting round (RESUME support)
    start_round = 0
    iter_mass_matrix = torch.load(MASS_MATRIX_PATH, weights_only=True)
    iter_step_size = adapted_step_size

    if RESUME:
        existing = sorted(ITER_DIR.glob("round_*.pt"))
        if existing:
            last_path = existing[-1]
            last = torch.load(last_path, weights_only=False)
            start_round = int(last["round"])
            iter_mass_matrix = last["mass_matrix"]
            iter_step_size = last["step_size"]
            print(f"Resuming from {last_path.name} (round {start_round})")

    print(
        f"H1-1: {N_ROUNDS} rounds × ({N_WARMUP_PER_ROUND} warmup + "
        f"{N_SAMPLES_PER_ROUND} production)"
    )
    print(f"Starting at round {start_round + 1}")


H1-1: 5 rounds × (2000 warmup + 500 production)
Starting at round 1


In [14]:
# ── H1-1 iterative loop ──
if RUN_ITERATIVE:
    round_results = []

    # Preload existing summary if resuming
    summary_path = ITER_DIR / "round_summary_raw.json"
    if RESUME and summary_path.exists():
        with open(summary_path) as f:
            round_results = _json.load(f)

    for rnd in range(start_round, N_ROUNDS):
        print(f"\n{'=' * 70}")
        print(f"ROUND {rnd + 1}/{N_ROUNDS}")
        print(f"{'=' * 70}")

        # Warmup: adapt M
        warmup_r = Sampler(bn, x_data, y_data).sample(
            config=NUTS(
                n_warmup=N_WARMUP_PER_ROUND,
                step_size=iter_step_size,
                max_tree_depth=MAX_TREE_DEPTH,
                target_accept=TARGET_ACCEPT,
                mass_matrix=iter_mass_matrix,
                adapt_mass_matrix=True,
            ),
            n_samples=1,
            n_chains=1,
            init_strategy=Perturb(scale=0.1),
            n_cores=1,
        )

        diag_w = warmup_r.chains[0].diagnostics
        new_M = diag_w.get("adapted_mass_matrix")
        new_eps = diag_w.get("adapted_step_size", diag_w["step_size"])
        if new_M is None:
            print(f"  ⚠ Round {rnd + 1}: no adapted M returned, stopping.")
            break
        iter_mass_matrix = new_M
        iter_step_size = new_eps

        # Production: fixed M
        prod_r = Sampler(bn, x_data, y_data).sample(
            config=NUTS(
                n_warmup=0,
                step_size=iter_step_size,
                max_tree_depth=MAX_TREE_DEPTH,
                target_accept=TARGET_ACCEPT,
                mass_matrix=iter_mass_matrix,
                adapt_mass_matrix=False,
            ),
            n_samples=N_SAMPLES_PER_ROUND,
            n_chains=1,
            init_strategy=Perturb(scale=0.1),
            n_cores=1,
        )

        samps_r = torch.stack(prod_r.chains[0].parameters).cpu().float()
        diag_p = prod_r.chains[0].diagnostics
        M_cpu = iter_mass_matrix.detach().cpu().float()
        sv = torch.linalg.svdvals(M_cpu)

        round_info = {
            "round": rnd + 1,
            "step_size": float(iter_step_size),
            "cond_M": float(sv[0] / sv[-1]) if sv[-1] > 0 else float("inf"),
            "acceptance_rate": float(prod_r.chains[0].acceptance_rate),
            "n_divergences": int(diag_p.get("n_divergences", 0)),
            "mean_tree_depth": float(diag_p.get("mean_tree_depth", 0)),
            "n_samples": int(samps_r.shape[0]),
        }
        round_results.append(round_info)

        # Per-round checkpoint (analysis recomputes ACF/ESS from samples + hessian.pt)
        torch.save(
            {
                "round": rnd + 1,
                "mass_matrix": iter_mass_matrix.cpu(),
                "step_size": float(iter_step_size),
                "samples": samps_r,
                "diagnostics": round_info,
            },
            ITER_DIR / f"round_{rnd + 1:02d}.pt",
        )

        with open(summary_path, "w") as f:
            _json.dump(round_results, f, indent=2)

        print(
            f"  ε={iter_step_size:.4e}  cond(M)={round_info['cond_M']:.2e}  "
            f"accept={round_info['acceptance_rate']:.3f}  "
            f"divs={round_info['n_divergences']}  "
            f"mean_depth={round_info['mean_tree_depth']:.1f}"
        )
        print(f"  ✓ Saved round_{rnd + 1:02d}.pt + round_summary_raw.json")

    print(f"\n{'=' * 70}")
    print(f"H1-1 complete: {len(round_results)} rounds total")
    print(f"Checkpoints in {ITER_DIR}")



ROUND 1/5


  [NUTS warmup] step 10/2001  ε=1.16e-01  depth=3  L=10  α=0.59  divs=4/10  mass=full


  [NUTS warmup] step 20/2001  ε=1.97e-02  depth=4 (hit max)  L=15  α=0.64  divs=2/10  mass=full


  [NUTS warmup] step 30/2001  ε=7.87e-02  depth=3  L=9  α=0.55  divs=1/10  mass=full


  [NUTS warmup] step 40/2001  ε=4.04e-03  depth=4 (hit max)  L=15  α=0.55  divs=0/10  mass=full


...

  [NUTS sample] step 460/500  ε=4.24e-03  depth=4 (hit max)  L=15  α=1.00  divs=0/10  mass=full


  [NUTS sample] step 470/500  ε=4.24e-03  depth=4 (hit max)  L=15  α=1.00  divs=0/10  mass=full


  [NUTS sample] step 480/500  ε=4.24e-03  depth=4 (hit max)  L=15  α=1.00  divs=0/10  mass=full


  [NUTS sample] step 490/500  ε=4.24e-03  depth=4 (hit max)  L=15  α=1.00  divs=0/10  mass=full


  [NUTS sample] step 500/500  ε=4.24e-03  depth=4 (hit max)  L=15  α=1.00  divs=0/10  mass=full
  ε=4.2385e-03  cond(M)=8.53e+10  accept=1.000  divs=0  mean_depth=4.0
  ✓ Saved round_05.pt + round_summary_raw.json

H1-1 complete: 5 rounds total
Checkpoints in /Users/ashrafahmed/code/slt-deep/projects/markov-chain-learning/experiments/single-chain/data/results_no_hessian/default/iterative
